In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/zainulabedeen589/mlproject.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="zainulabedeen589"
os.environ["MLFLOW_TRACKING_PASSWORD"]="0199657f9c6f275d19a81df463159045839c0bba"

In [2]:
import os
%pwd

'/Users/Apple/ZAINUL_AI/mlflow/ml_1/research'

In [3]:
os.chdir("../")
%pwd

'/Users/Apple/ZAINUL_AI/mlflow/ml_1'

In [12]:
from dataclasses import dataclass
from pathlib import Path


@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metrics_file_name: Path
    target_column: str
    mlflow_uri: str

In [17]:
from src.datascience.constants import *
from src.datascience.utils.common import (
    read_yaml,
    create_directories,
    save_json
)


class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:

        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            test_data_path=Path(config.test_data_path),
            model_path=Path(config.model_path),
            all_params=params,
            metrics_file_name=Path(config.metrics_file_name),
            target_column=schema,
            mlflow_uri="https://dagshub.com/zainulabedeen589/mlproject.mlflow"
        )

        return model_evaluation_config

In [18]:
import os
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from pathlib import Path
from urllib.parse import urlparse

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

In [19]:
class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):

        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)

        return rmse, mae, r2

    def log_into_mlflow(self):

        # Load test dataset
        test_data = pd.read_csv(
            self.config.test_data_path
        )

        # Load trained model
        model = joblib.load(
            self.config.model_path
        )

        # Separate features and target
        test_x = test_data.drop(
            [self.config.target_column],
            axis=1
        )

        test_y = test_data[self.config.target_column]

        # Configure MLflow
        mlflow.set_tracking_uri(
            self.config.mlflow_uri
        )

        mlflow.set_registry_uri(
            self.config.mlflow_uri
        )

        tracking_url_type_store = urlparse(
            mlflow.get_tracking_uri()
        ).scheme

        # Evaluate and log results
        with mlflow.start_run():

            predictions = model.predict(test_x)

            rmse, mae, r2 = self.eval_metrics(
                test_y,
                predictions
            )

            # Print evaluation metrics
            print(f"RMSE: {rmse}")
            print(f"MAE: {mae}")
            print(f"R2 Score: {r2}")

            # Save metrics locally
            scores = {
                "rmse": float(rmse),
                "mae": float(mae),
                "r2": float(r2)
            }

            metrics_path = Path(
                self.config.metrics_file_name
            )

            create_directories([metrics_path.parent])

            save_json(
                path=metrics_path,
                data=scores
            )

            # Log parameters
            mlflow.log_params(
                dict(self.config.all_params)
            )

            # Log metrics
            mlflow.log_metric("rmse", float(rmse))
            mlflow.log_metric("mae", float(mae))
            mlflow.log_metric("r2", float(r2))

            # Log model
            if tracking_url_type_store != "file":

                mlflow.sklearn.log_model(
                    sk_model=model,
                    artifact_path="model",
                    registered_model_name="ElasticnetModel"
                )

            else:

                mlflow.sklearn.log_model(
                    sk_model=model,
                    artifact_path="model"
                )

            print("Model evaluation completed successfully.")

In [20]:
try:
    config = ConfigurationManager()

    model_evaluation_config = (
        config.get_model_evaluation_config()
    )

    print("Model evaluation configuration loaded successfully.")

    model_evaluation = ModelEvaluation(
        config=model_evaluation_config
    )

    model_evaluation.log_into_mlflow()

except Exception as e:
    print(f"Error during model evaluation: {e}")
    raise

[2026-10-23 18:54:49,451: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-10-23 18:54:49,453: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-23 18:54:49,455: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-10-23 18:54:49,456: INFO: common: created directory at: artifacts]
[2026-10-23 18:54:49,456: INFO: common: created directory at: artifacts/model_evaluation]
Model evaluation configuration loaded successfully.
RMSE: 0.6782324763388069
MAE: 0.5393036211884986
R2 Score: 0.2513032357410966
[2026-10-23 18:54:57,355: INFO: common: created directory at: artifacts/model_evaluation]
[2026-10-23 18:54:57,356: INFO: common: json file saved at: artifacts/model_evaluation/metrics.json]


2026/10/23 18:55:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/23 18:55:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'ElasticnetModel'.
2026/10/23 18:55:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


Model evaluation completed successfully.
🏃 View run indecisive-ant-786 at: https://dagshub.com/zainulabedeen589/mlproject.mlflow/#/experiments/0/runs/a8ce2197daeb4bc49265d1fb99dad335
🧪 View experiment at: https://dagshub.com/zainulabedeen589/mlproject.mlflow/#/experiments/0
